**第 2 章 —— 端到端機器學習專案（End-to-End Machine Learning Project）**

*本筆記本包含《Hands-On Machine Learning》第 2 章的所有範例程式碼與習題解答，並已補充繁體中文說明與程式註解，方便碩士班同學課堂學習與自修使用。*

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/hanchen92/THU-STAT-BigData-115-1-6200/blob/main/02_end_to_end_machine_learning_project_zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/hanchen92/THU-STAT-BigData-115-1-6200/blob/main/02_end_to_end_machine_learning_project_zh.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

In [ ]:
# 匯入本筆記本後續會用到的基礎套件
import sklearn   # Scikit-Learn：機器學習模型與工具的主要套件
import numpy as np  # NumPy：數值運算套件

# 資料蒐集（Data Collection）

*歡迎加入「機器學習房屋公司」！你的任務是根據加州各行政區（district）的一些特徵，預測該地區房屋的中位數價值（median house value）。*

## 下載資料

In [ ]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    """下載並讀取加州房價資料集，若本機已有快取檔案則直接讀取，避免重複下載。"""
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        # 第一次執行時，本機還沒有資料，於是建立 datasets 資料夾並下載壓縮檔
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

## 快速瀏覽資料結構

In [ ]:
housing.head()  # 檢視資料集的前 5 筆資料，快速了解各欄位長相

In [ ]:
housing.info()  # 檢視每個欄位的資料型別與非缺失值筆數，可用來快速找出有缺失值的欄位

In [ ]:
housing["ocean_proximity"].value_counts()  # 這是一個類別型欄位，檢視各類別出現的次數

In [ ]:
housing.describe()  # 針對所有數值型欄位，輸出平均數、標準差、四分位數等敘述統計量

以下這個 cell 在原書中並未列出。它會建立 `images/end_to_end_project` 資料夾（若尚不存在的話），並定義 `save_fig()` 函式，本筆記本後續會用這個函式，把圖表存成高解析度的圖片檔。

In [ ]:
# extra code —— 用來把圖表存成高解析度 PNG 檔的輔助程式碼
IMAGES_PATH = Path() / "images" / "end_to_end_project"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    """將目前的圖表存成檔案，方便日後在報告或投影片中使用。"""
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [ ]:
import matplotlib.pyplot as plt

# extra code —— 以下 5 行是在設定圖表預設的文字大小
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

# 針對每個數值型欄位畫出直方圖，觀察其分布形狀（例如是否有偏態、長尾等現象）
housing.hist(bins=50, figsize=(12, 8))
save_fig("attribute_histogram_plots")  # extra code
plt.show()

## 建立測試集

### 簡單隨機抽樣（Simple Random Sampling）

In [ ]:
from sklearn.model_selection import train_test_split

# 最簡單的做法：直接把 20% 的資料隨機抽出來當測試集
# random_state=42 是為了固定亂數種子，確保每次執行結果一致，方便重現
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

In [ ]:
test_set["total_bedrooms"].isnull().sum()  # 檢查測試集中，total_bedrooms 欄位有幾筆缺失值

### 分層抽樣（Stratified Sampling）

In [ ]:
# 由於 median_income（收入中位數）是預測房價的重要特徵，
# 為了讓訓練集/測試集的收入分布與母體一致，我們先把收入切成幾個「收入類別」區間
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

In [ ]:
# 畫出各收入類別的樣本數，確認每個類別都有足夠的樣本數可供分層抽樣
housing["income_cat"].value_counts().sort_index().plot.bar(rot=0, grid=True)
plt.xlabel("Income category")
plt.ylabel("Number of districts")
save_fig("housing_income_cat_bar_plot")  # extra code
plt.show()

In [ ]:
# 以 income_cat 作為分層依據，確保訓練集與測試集中，各收入類別的比例與母體一致
strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42)

In [ ]:
strat_test_set["income_cat"].value_counts() / len(strat_test_set)  # 檢視分層測試集中各收入類別的比例

### 簡單隨機抽樣 vs. 分層抽樣 的比較

In [ ]:
# extra code —— 計算並比較「母體」「分層抽樣測試集」「簡單隨機抽樣測試集」
#              三者在各收入類別上的比例差異

def income_cat_proportions(data):
    return data["income_cat"].value_counts() / len(data)

train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

compare_props = pd.DataFrame({
    "Overall %": income_cat_proportions(housing),
    "Stratified %": income_cat_proportions(strat_test_set),
    "Random %": income_cat_proportions(test_set),
}).sort_index()
compare_props.index.name = "Income Category"
compare_props["Strat. Error %"] = (compare_props["Stratified %"] /
                                   compare_props["Overall %"] - 1)
compare_props["Rand. Error %"] = (compare_props["Random %"] /
                                  compare_props["Overall %"] - 1)
(compare_props * 100).round(2)
# 觀察重點：分層抽樣（Strat. Error %）的誤差通常會遠小於簡單隨機抽樣（Rand. Error %）

In [ ]:
# income_cat 只是為了做分層抽樣而暫時新增的輔助欄位，抽樣完成後就可以刪除，還原成原本的資料集
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# 資料探索（Data Exploration）

In [ ]:
# 從現在開始，我們只在「訓練集」上進行資料探索與分析，
# 測試集要留到最後才拿出來評估最終模型，避免「資料窺探偏誤」(data snooping bias)
housing = strat_train_set.copy()

## 視覺化地理資料

下一個 cell 會產生本章的第一張圖（此程式碼並未列在書中）。它只是前一張散佈圖的美化版本：加上加州地圖作為背景、較好看的座標標籤名稱，並移除格線。

In [ ]:
# extra code —— 這個 cell 用來產生本章的第一張圖

# 下載加州地圖的背景圖片
filename = "california.png"
if not (IMAGES_PATH / filename).is_file():
    homl3_root = "https://github.com/ageron/handson-ml3/raw/main/"
    url = homl3_root + "images/end_to_end_project/" + filename
    print("Downloading", filename)
    urllib.request.urlretrieve(url, IMAGES_PATH / filename)

housing_renamed = housing.rename(columns={
    "latitude": "Latitude", "longitude": "Longitude",
    "population": "Population",
    "median_house_value": "Median house value (ᴜsᴅ)"})
# 用經緯度畫散佈圖：點的大小代表人口數，顏色代表房價中位數
housing_renamed.plot(
             kind="scatter", x="Longitude", y="Latitude",
             s=housing_renamed["Population"] / 100, label="Population",
             c="Median house value (ᴜsᴅ)", cmap="jet", colorbar=True,
             legend=True, sharex=False, figsize=(10, 7))

california_img = plt.imread(IMAGES_PATH / filename)
axis = -124.55, -113.95, 32.45, 42.05
plt.axis(axis)
plt.imshow(california_img, extent=axis)  # 把加州地圖疊在散佈圖背後

save_fig("california_housing_prices_plot")
plt.show()

## 尋找相關性

注意：自 Pandas 2.0.0 起，`numeric_only` 參數預設值改為 `False`，因此我們需要明確將它設為 `True`，才能避免錯誤。

In [ ]:
corr_matrix = housing.corr(numeric_only=True)  # 計算各數值欄位兩兩之間的皮爾森相關係數

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)
# 由高到低檢視「各欄位」與「房價中位數」的相關係數，藉此了解哪些特徵可能最有預測力

In [ ]:
from pandas.plotting import scatter_matrix

# 選出幾個相關性較高的欄位，用散佈圖矩陣一次觀察它們兩兩之間的關係
attributes = ["median_house_value", "median_income", "total_rooms",
              "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12, 8))
save_fig("scatter_matrix_plot")  # extra code
plt.show()

In [ ]:
# 放大檢視「收入中位數」與「房價中位數」的關係，這是目前看起來最有希望的預測特徵
housing.plot(kind="scatter", x="median_income", y="median_house_value",
             alpha=0.1, grid=True)
save_fig("income_vs_house_value_scatterplot")  # extra code
plt.show()

# 特徵工程（Feature Engineering）

In [ ]:
# 嘗試組合出幾個「比例型」的新特徵，通常比原始欄位更有解釋力
housing["rooms_per_house"] = housing["total_rooms"] / housing["households"]        # 平均每戶的房間數
housing["bedrooms_ratio"] = housing["total_bedrooms"] / housing["total_rooms"]      # 臥室佔總房間數的比例
housing["people_per_house"] = housing["population"] / housing["households"]         # 平均每戶的人口數

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)
# 觀察新增的比例型特徵，是否比原始欄位（如 total_rooms、households）與房價的相關性更高

## 自訂轉換器（Custom Transformers）

讓我們回到原始的訓練集，並把目標變數（label）分離出來（注意：`strat_train_set.drop()` 會建立一份「移除該欄位」的副本，並不會真的修改 `strat_train_set` 本身，除非你傳入 `inplace=True`）：

In [ ]:
housing = strat_train_set.drop("median_house_value", axis=1)  # 特徵（不含目標欄位）
housing_labels = strat_train_set["median_house_value"].copy()   # 目標變數（要預測的房價）

以下示範如何建立簡單的轉換器（Transformer）：

In [ ]:
from sklearn.preprocessing import FunctionTransformer

# FunctionTransformer 可以把任意函式包裝成符合 Scikit-Learn 介面的轉換器，
# inverse_func 則可以在需要「還原」時使用（例如把預測結果轉換回原始尺度）
log_transformer = FunctionTransformer(np.log, inverse_func=np.exp)
log_pop = log_transformer.transform(housing[["population"]])

RBF 核函數（Radial Basis Function Kernel，高斯核函數）
$$k(x,y)=\exp{(-\gamma\|x-y\|^2_2)}$$

此函數可用來衡量兩點 $x, y$ 之間的「相似度」：距離越近，相似度越接近 1；距離越遠，相似度越接近 0。
參數 $\gamma$ 控制相似度隨距離增加而下降的速度，$\gamma$ 越大，相似度衰減得越快。

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

# 計算每個地區的「房屋屋齡中位數」與 35 年之間的相似度
age_simil_35 = rbf_kernel(housing[["housing_median_age"]], [[35]], gamma=0.1)

In [ ]:
# extra code —— 這個 cell 用來產生 Figure 2–18

ages = np.linspace(housing["housing_median_age"].min(),
                   housing["housing_median_age"].max(),
                   500).reshape(-1, 1)
gamma1 = 0.1
gamma2 = 0.03
rbf1 = rbf_kernel(ages, [[35]], gamma=gamma1)
rbf2 = rbf_kernel(ages, [[35]], gamma=gamma2)

fig, ax1 = plt.subplots()

ax1.set_xlabel("Housing median age")
ax1.set_ylabel("Number of districts")
ax1.hist(housing["housing_median_age"], bins=50)

ax2 = ax1.twinx()  # 建立一個共用 x 軸、但擁有獨立 y 軸的副座標軸
color = "blue"
ax2.plot(ages, rbf1, color=color, label="gamma = 0.10")
ax2.plot(ages, rbf2, color=color, label="gamma = 0.03", linestyle="--")
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylabel("Age similarity", color=color)
# 觀察重點：gamma 越大（實線），相似度隨屋齡差距增加而下降得越快，形狀也越「尖」

plt.legend(loc="upper left")
save_fig("age_similarity_plot")
plt.show()

In [ ]:
# 計算每個地區與「舊金山」這個地點之間的地理相似度（同樣用 RBF 核函數）
sf_coords = 37.7749, -122.41
sf_transformer = FunctionTransformer(rbf_kernel,
                                     kw_args=dict(Y=[sf_coords], gamma=0.1))
sf_simil = sf_transformer.transform(housing[["latitude", "longitude"]])

In [ ]:
sf_simil  # 檢視每個地區與舊金山之間的相似度數值

In [ ]:
# 用 FunctionTransformer 包裝一個「取比例」的轉換器：第 0 欄除以第 1 欄
ratio_transformer = FunctionTransformer(lambda X: X[:, [0]] / X[:, [1]])
ratio_transformer.transform(np.array([[1., 2.], [3., 4.]]))

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

# 自行實作一個「簡化版」的 StandardScaler，示範如何撰寫符合 Scikit-Learn 介面的自訂轉換器
class StandardScalerClone(BaseEstimator, TransformerMixin):
    def __init__(self, with_mean=True):  # 建構子中不可使用 *args 或 **kwargs！
        self.with_mean = with_mean

    def fit(self, X, y=None):  # 即使用不到 y，也一定要保留這個參數
        X = check_array(X)  # 檢查 X 是否為數值有限的陣列
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.n_features_in_ = X.shape[1]  # 每個 estimator 都應在 fit() 中記錄輸入特徵數
        return self  # fit() 一定要回傳 self！

    def transform(self, X):
        check_is_fitted(self)  # 檢查是否已經呼叫過 fit()（尋找結尾為 _ 的已學習屬性）
        X = check_array(X)
        assert self.n_features_in_ == X.shape[1]
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_

In [ ]:
from sklearn.cluster import KMeans

# 自訂一個「地理群集相似度」轉換器：
# 先用 KMeans 把地區依經緯度分成幾個群集，再計算每個地區與各群集中心的 RBF 相似度
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # fit() 一定要回傳 self！

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

**注意事項**：

* Scikit-Learn 1.3.0 版之後，`KMeans` 初始化所使用的亂數產生器有所變動，因此若你使用 Scikit-Learn ≥ 1.3，
  執行結果可能會和書中不完全相同，這是正常現象，並不代表程式有誤。
* 本筆記本中，只要建立 `KMeans` 物件時沒有明確指定 `n_init`，都會明確設為 `n_init=10`，
  以避免因為此超參數預設值即將從 10 改為 `"auto"`（Scikit-Learn 1.4 起）而跳出警告訊息。

In [ ]:
# 將所有地區依經緯度分成 10 群，並計算每個地區與各群中心的相似度
# sample_weight=housing_labels：讓房價較高的地區在決定群集中心時，有較高的權重
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
similarities = cluster_simil.fit_transform(housing[["latitude", "longitude"]],
                                           sample_weight=housing_labels)

In [ ]:
similarities[:3].round(2)  # 檢視前 3 個地區，與 10 個群集中心的相似度

In [ ]:
# extra code —— 這個 cell 用來產生 Figure 2–19
fig, axes= plt.subplots(1, 2, figsize=(15, 5))

housing_renamed = housing.rename(columns={
    "latitude": "Latitude", "longitude": "Longitude",
    "population": "Population",
    "median_house_value": "Median house value (ᴜsᴅ)"})
housing_renamed["Max cluster similarity"] = similarities.max(axis=1)

# 左圖：原始的房價地理分布圖
strat_train_set.plot(kind="scatter", x="longitude", y="latitude", grid=True,
             s=housing_renamed["Population"] / 100, label="population",
             c="median_house_value", cmap="jet", colorbar=True,
             legend=True,ax = axes[0])

# 右圖：每個地區與「最相似群集」之間的相似度，並標出群集中心位置
housing_renamed.plot(kind="scatter", x="Longitude", y="Latitude", grid=True,
                     s=housing_renamed["Population"] / 100, label="Population",
                     c="Max cluster similarity",
                     cmap="jet", colorbar=True,
                     legend=True, sharex=False, ax= axes[1])
axes[1].plot(cluster_simil.kmeans_.cluster_centers_[:, 1],
         cluster_simil.kmeans_.cluster_centers_[:, 0],
         linestyle="", color="black", marker="X", markersize=20,
         label="Cluster centers")
axes[1].legend(loc="upper right")
#save_fig("district_cluster_plot")
#plt.show()

# 資料清理（Data Cleaning）

## 缺失值處理

處理缺失值（NaN）常見有三種做法：

```python
# 做法一
housing.dropna(subset=["total_bedrooms"], inplace=True)
# 做法二
housing.drop("total_bedrooms", axis=1, inplace=True)
# 做法三
median = housing["total_bedrooms"].median()
housing["total_bedrooms"].fillna(median, inplace=True)
```

針對每一種做法，我們都會先複製一份 `housing`，在複製品上操作，避免破壞原始的 `housing`。
同時也會展示每種做法的結果，並只篩選出「原本含有 NaN」的那些列，方便互相比較。

In [ ]:
null_rows_idx = housing.isnull().any(axis=1)  # 找出「至少有一個欄位缺失」的資料列
housing.loc[null_rows_idx].head()

做法一：刪除含有缺失值的資料列

In [ ]:
housing_option1 = housing.copy()

housing_option1.dropna(subset=["total_bedrooms"], inplace=True)  # 做法一

housing_option1.loc[null_rows_idx].head()  # 這些列已經被刪除，因此這裡會是空的

做法二：直接刪除含有缺失值的整個特徵欄位

In [ ]:
housing_option2 = housing.copy()

housing_option2.drop("total_bedrooms", axis=1, inplace=True)  # 做法二

housing_option2.loc[null_rows_idx].head()  # total_bedrooms 欄位已經整欄消失

做法三：以中位數填補缺失值

In [ ]:
housing_option3 = housing.copy()

median = housing["total_bedrooms"].median()
housing_option3["total_bedrooms"].fillna(median, inplace=True)  # 做法三

housing_option3.loc[null_rows_idx].head()  # 原本的 NaN 已經被中位數取代

In [ ]:
from sklearn.impute import SimpleImputer

# SimpleImputer 是 Scikit-Learn 內建的填補工具，這裡指定用「中位數」策略
imputer = SimpleImputer(strategy="median")

先把數值型的特徵欄位獨立出來，才能套用 `"median"`（中位數）填補策略（因為中位數無法計算在像 `ocean_proximity` 這種文字型特徵上）：

In [ ]:
housing_num = housing.select_dtypes(include=[np.number])  # 只挑出數值型欄位

In [ ]:
imputer.fit(housing_num)  # 讓 imputer 學習每個欄位的中位數

In [ ]:
imputer.statistics_  # 檢視 imputer 學到的每個欄位中位數

驗證一下，這是否和我們手動計算每個特徵中位數的結果相同：

In [ ]:
housing_num.median().values  # 手動計算的中位數，應該和 imputer.statistics_ 完全一致

將訓練集做轉換：

In [ ]:
X = imputer.transform(housing_num)  # 用學到的中位數，填補所有缺失值，回傳為 NumPy 陣列

In [ ]:
imputer.feature_names_in_  # 檢視 imputer 訓練時記住的欄位名稱與順序

In [ ]:
# 把填補後的 NumPy 陣列，重新包裝回帶有欄位名稱與索引的 DataFrame，方便閱讀
housing_tr = pd.DataFrame(X, columns=housing_num.columns,
                          index=housing_num.index)

In [ ]:
housing_tr.loc[null_rows_idx].head()  # 確認原本缺失的欄位，現在都已經被中位數填補

In [ ]:
imputer.strategy  # 檢視目前使用的填補策略

In [ ]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns,
                          index=housing_num.index)

In [ ]:
housing_tr.loc[null_rows_idx].head()  # 書中未展示這一段，這裡只是重複確認結果

## 類別型變數編碼（Encoding Categorical Variables）

接下來我們要處理類別型的輸入特徵 `ocean_proximity`（離海洋的距離類別）：

In [ ]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(8)

### 序數編碼（Ordinal Encoding）

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# 把每個類別依序對應到一個整數（例如 0, 1, 2, ...）
ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)

In [ ]:
housing_cat_encoded[:8]  # 檢視編碼後的結果（純數字，沒有大小順序上的意義）

In [ ]:
ordinal_encoder.categories_  # 檢視每個整數分別對應到哪一個原始類別

### 獨熱編碼（One-Hot Encoding）

**為什麼不直接用序數編碼？** 因為序數編碼會讓模型誤以為類別之間存在「大小關係」（例如誤以為類別 2 比類別 1「大」兩倍），但 `ocean_proximity` 的各類別之間其實沒有這種順序關係。獨熱編碼則會把每個類別轉成一個獨立的 0/1 欄位，避免這個問題。

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

In [ ]:
housing_cat_1hot  # 這是一個稀疏矩陣（大部分元素都是 0，只有少數是 1）

`OneHotEncoder` 預設會回傳一個稀疏矩陣（sparse array），若有需要，可以呼叫 `toarray()` 方法將其轉換成一般的密集矩陣（dense array）：

In [ ]:
housing_cat_1hot.toarray()

另一種做法是在建立 `OneHotEncoder` 時直接設定 `sparse_output=False`（注意：`sparse` 這個超參數在 Scikit-Learn 1.2 版之後改名為 `sparse_output`）：

In [ ]:
cat_encoder = OneHotEncoder(sparse_output=False)
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

In [ ]:
cat_encoder.categories_

如果直接對測試集資料做編碼，可能會產生和訓練集不同的欄位（因為測試集裡出現的類別，可能與訓練集不完全相同）。

In [ ]:
df_test = pd.DataFrame({"ocean_proximity": ["INLAND", "NEAR BAY"]})
pd.get_dummies(df_test)  # 只有出現在 df_test 裡的類別，才會被轉成欄位

In [ ]:
cat_encoder.transform(df_test)
# 相較之下，已訓練好的 cat_encoder 會記住「全部」類別，欄位數量固定，不會因輸入資料而改變

如果測試集中出現了訓練集沒看過的類別值，我們可以透過 `handle_unknown` 參數來忽略它，避免程式出錯。

In [ ]:
df_test_unknown = pd.DataFrame({"ocean_proximity": ["<2H OCEAN", "ISLAND"]})
pd.get_dummies(df_test_unknown)

In [ ]:
cat_encoder.handle_unknown = "ignore"  # 遇到未知類別時，直接忽略（該列所有欄位皆為 0），而不是報錯
cat_encoder.transform(df_test_unknown)

In [ ]:
cat_encoder.feature_names_in_  # 訓練時看到的輸入欄位名稱

In [ ]:
cat_encoder.get_feature_names_out()  # 獨熱編碼後，輸出的每個欄位名稱

In [ ]:
df_output = pd.DataFrame(cat_encoder.transform(df_test_unknown),
                         columns=cat_encoder.get_feature_names_out(),
                         index=df_test_unknown.index)

In [ ]:
df_output  # 帶有清楚欄位名稱的獨熱編碼結果

## 特徵縮放（Feature Scaling）

### 正規化（Min-Max Scaling）
$$x^*=\frac{x-\min{(x)}}{\max{(x)}-\min{(x)}}$$

將數值線性縮放到固定範圍（例如 0 到 1，或 -1 到 1）之間。

In [ ]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
housing_num_min_max_scaled = min_max_scaler.fit_transform(housing_num)

### 標準化（Standardization）

$$x^*=\frac{x-\mu_x}{\sigma_x}$$

將數值轉換為平均數 0、標準差 1 的分布。標準化不會限制在固定範圍內，
但相較於正規化，對離群值（outlier）較不敏感，是實務上更常見的做法。

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
housing_num_std_scaled = std_scaler.fit_transform(housing_num)

### 對數轉換（Log Transformation）

當某個特徵的分布呈現「長尾」（許多小值加上少數極大值）時，取對數可以讓分布較接近常態，有助於後續模型的表現。

In [ ]:
# extra code —— 這個 cell 用來產生 Figure 2–17
fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
housing["population"].hist(ax=axs[0], bins=50)              # 原始的人口數分布（長尾）
housing["population"].apply(np.log).hist(ax=axs[1], bins=50)  # 取對數後的分布（較接近對稱）
axs[0].set_xlabel("Population")
axs[1].set_xlabel("Log of population")
axs[0].set_ylabel("Number of districts")
save_fig("long_tail_plot")
plt.show()

### 百分位數轉換（Percentile Transformation）

如果我們把每個數值都換成它所對應的百分位數，會發生什麼事？

In [ ]:
# extra code —— 只是為了示範轉換後會變成均勻分布
percentiles = [np.percentile(housing["median_income"], p)
               for p in range(1, 100)]
flattened_median_income = pd.cut(housing["median_income"],
                                 bins=[-np.inf] + percentiles + [np.inf],
                                 labels=range(1, 100 + 1))
flattened_median_income.hist(bins=50)
plt.xlabel("Median income percentile")
plt.ylabel("Number of districts")
plt.show()
# 注意：低於第 1 百分位數的收入會被標記為 1，高於第 99 百分位數的則標記為 100，
# 這就是為什麼下方分布的範圍是 1 到 100（而不是 0 到 100）。

**為什麼要對目標變數（label）做縮放？** 有些模型（如線性迴歸）在目標變數的尺度差異很大時，訓練過程可能不太穩定。以下示範：先把目標變數標準化後再訓練，預測完再用 `inverse_transform` 還原回原始尺度。

In [ ]:
from sklearn.linear_model import LinearRegression

target_scaler = StandardScaler()
scaled_labels = target_scaler.fit_transform(housing_labels.to_frame())

model = LinearRegression()
model.fit(housing[["median_income"]], scaled_labels)
some_new_data = housing[["median_income"]].iloc[:5]  # 假裝這是全新的資料

scaled_predictions = model.predict(some_new_data)
predictions = target_scaler.inverse_transform(scaled_predictions)  # 把預測結果換回原始的房價尺度

In [ ]:
predictions

In [ ]:
from sklearn.compose import TransformedTargetRegressor

# TransformedTargetRegressor 可以把「目標變數縮放 + 還原」的流程自動化，
# 不需要像上面那樣手動呼叫 inverse_transform
model = TransformedTargetRegressor(LinearRegression(),
                                   transformer=StandardScaler())
model.fit(housing[["median_income"]], housing_labels)
predictions = model.predict(some_new_data)

In [ ]:
predictions

## 轉換管線（Transformation Pipelines）

現在讓我們建立一個管線（Pipeline），把「填補缺失值」與「標準化」兩個步驟串接起來：

In [ ]:
from sklearn.pipeline import Pipeline

# Pipeline 會依序執行每個步驟：先用中位數填補缺失值，再做標準化
num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

In [ ]:
from sklearn.pipeline import make_pipeline

# make_pipeline 是 Pipeline 的簡化寫法，會自動幫每個步驟命名
num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

In [ ]:
from sklearn import set_config

set_config(display='diagram')  # 讓 pipeline 以互動式圖表呈現，方便檢視結構

num_pipeline

In [ ]:
housing_num_prepared = num_pipeline.fit_transform(housing_num)  # 一次完成填補 + 標準化
housing_num_prepared[:2].round(2)

In [ ]:
# 把處理後的 NumPy 陣列，重新包裝回帶有欄位名稱的 DataFrame，方便閱讀
df_housing_num_prepared = pd.DataFrame(
    housing_num_prepared, columns=num_pipeline.get_feature_names_out(),
    index=housing_num.index)

In [ ]:
df_housing_num_prepared.head(2)  # extra code

In [ ]:
num_pipeline.steps  # 檢視 pipeline 中所有步驟的名稱與物件

In [ ]:
num_pipeline[1]  # 用索引取出 pipeline 中的第 2 個步驟（索引從 0 開始）

In [ ]:
num_pipeline[:-1]  # 取出「除了最後一個步驟」以外的所有步驟

In [ ]:
num_pipeline.named_steps["simpleimputer"]  # 用步驟名稱取出對應的轉換器物件

In [ ]:
num_pipeline.set_params(simpleimputer__strategy="median")  # 用「步驟名稱__參數名稱」的語法調整超參數

In [ ]:
from sklearn.compose import ColumnTransformer

# 分別列出數值型欄位與類別型欄位
num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
cat_attribs = ["ocean_proximity"]

# 類別型欄位的管線：先用眾數填補缺失值，再做獨熱編碼
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"))

# ColumnTransformer 可以讓「不同欄位」套用「不同的前處理管線」，
# 這裡數值型欄位走 num_pipeline，類別型欄位走 cat_pipeline，最後再自動合併結果
preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer

# 更簡潔的寫法：用 make_column_selector 依「資料型別」自動挑選欄位，
# 不需要手動把每個欄位名稱都寫出來
preprocessing = make_column_transformer(
    (num_pipeline, make_column_selector(dtype_include=np.number)),
    (cat_pipeline, make_column_selector(dtype_include=object)),
)

In [ ]:
housing_prepared = preprocessing.fit_transform(housing)  # 一次完成所有欄位的前處理

In [ ]:
# extra code —— 示範如何把處理結果轉換回帶有欄位名稱的 DataFrame
housing_prepared_fr = pd.DataFrame(
    housing_prepared,
    columns=preprocessing.get_feature_names_out(),
    index=housing.index)
housing_prepared_fr.head(2)

### 線性迴歸

In [ ]:
from sklearn.linear_model import LinearRegression

# 用 make_pipeline 把「前處理」與「線性迴歸模型」串接成一個完整的模型，
# 之後只要呼叫 .fit() / .predict()，就會自動先做前處理，再交給模型訓練或預測
lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

讓我們在幾筆訓練樣本上，試跑完整的預處理管線：

In [ ]:
housing_predictions = lin_reg.predict(housing)
housing_predictions[:5].round(-2)  # -2 代表四捨五入到「百」的位數

與真實數值比較：

In [ ]:
housing_labels.iloc[:5].values

In [ ]:
# extra code —— 計算書中提到的誤差比例
error_ratios = housing_predictions[:5].round(-2) / housing_labels.iloc[:5].values - 1
print(", ".join([f"{100 * ratio:.1f}%" for ratio in error_ratios]))

In [ ]:
from sklearn.metrics import mean_squared_error

# 計算均方根誤差 RMSE，數值越小代表模型預測越準確（單位與房價相同，即美元）
lin_rmse = mean_squared_error(housing_labels, housing_predictions,
                              squared=False)
lin_rmse

### 決策樹（Decision Tree）

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)

In [ ]:
housing_predictions = tree_reg.predict(housing)
tree_rmse = mean_squared_error(housing_labels, housing_predictions,
                              squared=False)
tree_rmse
# 注意：這裡的 RMSE 很可能趨近於 0，代表決策樹幾乎「完美地」背下了訓練資料，
# 這通常是嚴重過度配適（overfitting）的警訊，而不是模型真的表現很好！

In [ ]:
from sklearn.model_selection import cross_val_score

# extra code —— 計算線性迴歸模型的交叉驗證誤差統計量
# cv=10 代表把訓練集切成 10 份，輪流用其中 9 份訓練、1 份驗證，重複 10 次
lin_rmses = -cross_val_score(lin_reg, housing, housing_labels,
                              scoring="neg_root_mean_squared_error", cv=10)
pd.Series(lin_rmses).describe()

In [ ]:
tree_rmses = -cross_val_score(tree_reg, housing, housing_labels,
                              scoring="neg_root_mean_squared_error", cv=10)

In [ ]:
pd.Series(tree_rmses).describe()
# 觀察重點：決策樹在交叉驗證上的表現，通常反而比線性迴歸差（甚至不穩定），
# 印證了它在訓練集上的「完美表現」只是過度配適，並非真正學到有用的規律

**注意：** 以下這個 cell 可能需要執行幾分鐘。

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# 隨機森林：由多棵決策樹組成的「集成模型」(ensemble)，通常比單一決策樹更穩健、更不易過度配適
forest_reg = make_pipeline(preprocessing,
                           RandomForestRegressor(random_state=42))
forest_rmses = -cross_val_score(forest_reg, housing, housing_labels,
                                scoring="neg_root_mean_squared_error", cv=10)

In [ ]:
pd.Series(forest_rmses).describe()

讓我們比較用交叉驗證算出的 RMSE（也就是「驗證誤差」），與直接在訓練集上算出的 RMSE（也就是「訓練誤差」）：

In [ ]:
forest_reg.fit(housing, housing_labels)
housing_predictions = forest_reg.predict(housing)
forest_rmse = mean_squared_error(housing_labels, housing_predictions,
                                 squared=False)
forest_rmse

訓練誤差明顯低於驗證誤差，這通常代表模型對訓練集發生了過度配適（overfitting）。另一種可能的解釋是訓練資料與驗證資料之間存在分布不一致（mismatch），但這裡並非如此，因為兩者都來自同一份資料集，只是經過洗牌後切分成兩部分。

**注意：** 以下這個 cell 可能需要執行幾分鐘。

In [ ]:
from sklearn.model_selection import GridSearchCV

# 把「前處理」與「隨機森林模型」包成一個完整的 pipeline，
# 這樣就能同時搜尋前處理的超參數（如群集數量）與模型本身的超參數
full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])
# param_grid 定義了要嘗試的超參數組合：
#   - preprocessing__geo__n_clusters：地理群集相似度轉換器中的群集數量
#   - random_forest__max_features：每次分裂節點時，隨機挑選的特徵數
param_grid = [
    {'preprocessing__geo__n_clusters': [5, 8, 10],
     'random_forest__max_features': [4, 6, 8]},
    {'preprocessing__geo__n_clusters': [10, 15],
     'random_forest__max_features': [6, 8, 10]},
]
grid_search = GridSearchCV(full_pipeline, param_grid, cv=3,
                           scoring='neg_root_mean_squared_error')
grid_search.fit(housing, housing_labels)  # 會嘗試所有組合，並各自做 3 折交叉驗證

你可以透過 `full_pipeline.get_params().keys()` 取得所有可供調整的超參數完整清單：

In [ ]:
# extra code —— 只顯示 get_params().keys() 輸出結果的一部分
print(str(full_pipeline.get_params().keys())[:1000] + "...")

找到的最佳超參數組合：

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_estimator_  # 使用最佳超參數訓練出來的完整 pipeline

讓我們檢視網格搜尋過程中，每一組超參數組合對應的分數：

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)

# extra code —— 以下幾行只是為了讓 DataFrame 看起來更清楚易讀
cv_res = cv_res[["param_preprocessing__geo__n_clusters",
                 "param_random_forest__max_features", "split0_test_score",
                 "split1_test_score", "split2_test_score", "mean_test_score"]]
score_cols = ["split0", "split1", "split2", "mean_test_rmse"]
cv_res.columns = ["n_clusters", "max_features"] + score_cols
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)

cv_res.head()